In [ ]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text


server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_sa
db_sql = "ODIN"
user_sql = user_sa
pwd_sql = pwd_sa
engine_sa = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

def nombre_mes_anio(fecha_mes_base):
    from datetime import datetime

    fecha = datetime.strptime(fecha_mes_base, "%Y-%m-%d")

    meses = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]

    return f"{meses[fecha.month - 1]} {fecha.year}"


In [ ]:
df_acumulado_cli.head()

,DNI,COD_SUCURSAL,SUCURSAL,MONTO,ASESOR,CANALVENTA,VENDOR_VERIFICADO,FECHA_DESEMBOLSO,CODIGO_ID
0,2969,8334,PUCALLPA,5100.0,E03973,Agencia,NaN,2026-07-07,E13418
1,16021,8334,PUCALLPA,3000.0,E11323,FFVV,ODISEC,2026-07-03,44727899
2,27595,8334,PUCALLPA,4000.0,E03973,Agencia,NaN,2026-07-04,NaN
3,43287,100,Central,9700.0,E04035,Banca Telefónica Comercial,NaN,2026-07-01,75431844
4,61822,8334,PUCALLPA,20000.0,E03973,FFVV,ODISEC,2026-07-07,71038496


In [ ]:
df_desembolso_cli.head()

,DNI,CUENTA_BT,N_OPER,COD_SUCURSAL,SUCURSAL,MONTO_FINANCIADO,FECHA_SOL,FECHA_DESEMBOLSOS,TEA,CANAL,TIPO_DESEM,CODIGO_ID
0,32841787,17916083,8463352,4272,CHIMBOTE,2000.0,2026-07-04,2026-07-04,70.0,TARGET,DERIVACION,1
1,8884832,18194112,8463275,4299,CUSCO LA CULTURA,14000.0,2026-07-03,2026-07-03,46.0,TARGET,DERIVACION,75283477
2,250811,18193760,8462888,2000,CAR - TUMBES,5000.0,2026-07-02,2026-07-02,65.0,TARGET,DERIVACION,77091562
3,21785243,18193835,8463042,8391,CHINCHA,2000.0,2026-07-02,2026-07-02,77.0,TARGET,DERIVACION,46271746


In [ ]:

filename='ACUM_DESEM.txt'

ruta_archivo = os.path.join(ruta_ventas_desembolso, filename)
df_acumulado_cli = pd.read_csv(ruta_archivo,sep='|')

df_acumulado_cli["FECHA_DESEMBOLSO"] = pd.to_datetime(df_acumulado_cli["FECHA_DESEMBOLSO"], errors="coerce")

fecha_min = df_acumulado_cli["FECHA_DESEMBOLSO"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_acumulado_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)


from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.Alfin_ventas_desembolso
        WHERE FECHA_DESEMBOLSO >= '{fecha_desembolso}'
          AND FECHA_DESEMBOLSO <= EOMONTH('{fecha_desembolso}');
    """))

df_acumulado_cli.to_sql(
    name="Alfin_ventas_desembolso",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)




2026-07-01 00:00:00


346

In [ ]:
filename='TARGET.txt'

ruta_archivo = os.path.join(ruta_ventas_desembolso, filename)
df_desembolso_varios_cli = pd.read_csv(ruta_archivo,sep='|')

df_desembolso_varios_cli["FECHA_DESEMBOLSOS"] = pd.to_datetime(df_desembolso_varios_cli["FECHA_DESEMBOLSOS"], errors="coerce")

fecha_min = df_desembolso_varios_cli["FECHA_DESEMBOLSOS"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_desembolso_varios_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)


from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.Alfin_ventas_desembolso_varios
        WHERE FECHA_DESEMBOLSOS >= '{fecha_desembolso}'
          AND FECHA_DESEMBOLSOS <= EOMONTH('{fecha_desembolso}');
    """))

df_desembolso_varios_cli.to_sql(
    name="Alfin_ventas_desembolso_varios",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)





2026-07-02 00:00:00


4